**11/21/2025**

##Converting Data from Xena TCGA-GBM Glioblastoma (GBM) Cohort into numerical values and saving in csv files for models

Datasets: Methylation27k (DNA methylation), Phenotype (Clinical Data + ground truths), HiSeqV2 (mRNA seq)


https://xenabrowser.net/datapages/?cohort=GDC%20TCGA%20Glioblastoma%20(GBM)&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443


Read the column names of the clinical data to find which one contains the subtypes

In [ ]:
import pandas as pd
from google.colab import files
uploaded = files.upload()

clin = pd.read_csv("TCGA.GBM.sampleMap_GBM_clinicalMatrix", sep="\t", index_col=0)
print(clin.columns.tolist())


Saving HiSeqV2 to HiSeqV2
Saving HumanMethylation27 to HumanMethylation27
Saving TCGA.GBM.sampleMap_GBM_clinicalMatrix to TCGA.GBM.sampleMap_GBM_clinicalMatrix
['CDE_DxAge', 'CDE_alk_chemoradiation_standard', 'CDE_chemo_adjuvant_alk', 'CDE_chemo_adjuvant_tmz', 'CDE_chemo_alk', 'CDE_chemo_alk_days', 'CDE_chemo_alk_long', 'CDE_chemo_tmz', 'CDE_chemo_tmz_days', 'CDE_chemo_tmz_long', 'CDE_missing', 'CDE_missingflag', 'CDE_previously_treated', 'CDE_radiation_adjuvant', 'CDE_radiation_adjuvant_standard', 'CDE_radiation_adjuvant_standard_probable', 'CDE_radiation_any', 'CDE_radiation_standard', 'CDE_radiation_standard_probable', 'CDE_sourcesite', 'CDE_survival_time', 'CDE_suspect', 'CDE_therapy', 'CDE_tmz_chemoradiation_standard', 'CDE_vital_status', 'G_CIMP_STATUS', 'GeneExp_Subtype', 'In_Cancer_Cell_Paper', '_INTEGRATION', '_PANCAN_CNA_PANCAN_K8', '_PANCAN_Cluster_Cluster_PANCAN', '_PANCAN_DNAMethyl_GBM', '_PANCAN_DNAMethyl_PANCAN', '_PANCAN_RPPA_PANCAN_K8', '_PANCAN_UNC_RNAseq_PANCAN_K16',

Load the three datasets

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif # f_classif is no longer used, but SelectKBest is fine if needed for future work
# from google.colab import files # Assuming files.upload() was run in the cell above

# -------------------------------
# LOAD RAW FILES (Assuming files were uploaded in the previous cell)
# -------------------------------
rna = pd.read_csv("HiSeqV2", sep="\t", index_col=0)
meth = pd.read_csv("HumanMethylation27", sep="\t", index_col=0)
clin = pd.read_csv("TCGA.GBM.sampleMap_GBM_clinicalMatrix", sep="\t", index_col=0)

# -------------------------------
# CLEAN SAMPLE IDS
# -------------------------------
def clean_barcode(x):
    # This should handle Series/Index cleaning
    if isinstance(x, pd.Index):
        return x.str[:12].str.upper()
    return x.str[:12].str.upper()

rna.columns = clean_barcode(rna.columns)
meth.columns = clean_barcode(meth.columns)
clin.index = clean_barcode(clin.index)

# -------------------------------
# DROP EMPTY / CONSTANT CLINICAL COLUMNS
# -------------------------------
def drop_empty(df, threshold=0.95):
    keep = df.isna().mean() < threshold
    df = df.loc[:, keep]
    nunique = df.nunique()
    df = df.loc[:, nunique > 1]
    return df

clin = drop_empty(clin)

# -------------------------------
# EXTRACT LABEL COLUMN
# -------------------------------
label_candidates = ["GeneExp_Subtype", "Subtype_Classification", "TCGA_Subtype",
                    "PAM50", "GeneExpression.Subtype", "Subtype"]

subtype_col = next((c for c in label_candidates if c in clin.columns), None)
if subtype_col is None:
    raise ValueError("No subtype column found in clinical matrix.")

labels = clin[subtype_col].astype(str)
clin = clin.drop(columns=[subtype_col])

# -------------------------------
# INTERSECT SAMPLES
# -------------------------------
common_samples = sorted(list(
    set(rna.columns) & set(meth.columns) & set(clin.index) & set(labels.index)
))

rna = rna[common_samples].T
meth = meth[common_samples].T
clin = clin.loc[common_samples]
labels = labels.loc[common_samples]

# -------------------------------
# REMOVE DUPLICATES
# -------------------------------
def remove_dupes(df, name):
    if df.index.duplicated().any():
        print(f"Removing duplicates in {name}: {df.index[df.index.duplicated()].tolist()}")
    return df[~df.index.duplicated(keep="first")]

rna = remove_dupes(rna, "RNA")
meth = remove_dupes(meth, "Methylation")
clin = remove_dupes(clin, "Clinical")
labels = labels[~labels.index.duplicated(keep="first")]

# -------------------------------
# REPLACE NaNs / INFs
# -------------------------------
# Use -999 for numerical fillna, and a placeholder string for categorical clinical data
rna = rna.replace([np.inf, -np.inf], np.nan).fillna(-999).astype(float)
meth = meth.replace([np.inf, -np.inf], np.nan).fillna(-999).astype(float)
clin = clin.replace([np.inf, -np.inf], np.nan).fillna("-999")

# -------------------------------
# RNA ENCODING (top 1000 variable genes)
# -------------------------------
top_n_genes = 1000
gene_var = rna.var(axis=0)
top_genes = gene_var.sort_values(ascending=False).head(top_n_genes).index
rna_selected = rna[top_genes]

rna_encoded = pd.DataFrame(
    StandardScaler().fit_transform(rna_selected),
    index=rna_selected.index,
    columns=rna_selected.columns
)
print(f"RNA features encoded: {rna_encoded.shape[1]}")

# -------------------------------
# METHYLATION ENCODING (Top 5000 Probes by Variance - FIXED LEAKAGE)
# -------------------------------
top_n_probes = 5000

# Use variance to select features, which is agnostic to the labels
# Selecting based on variance on the entire dataset is safe, as it doesn't use 'y'.
probe_var = meth.var(axis=0)
top_probes = probe_var.sort_values(ascending=False).head(top_n_probes).index

meth_selected = meth[top_probes] # Select top probes from the cleaned data

# Standardize the selected data
meth_encoded = pd.DataFrame(
    StandardScaler().fit_transform(meth_selected),
    index=meth_selected.index,
    columns=meth_selected.columns
)
print(f"Methylation features encoded: {meth_encoded.shape[1]}")

# -------------------------------
# CLINICAL ENCODING
# -------------------------------
numeric_cols = clin.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = clin.select_dtypes(exclude=[np.number]).columns.tolist()

clin_num = pd.DataFrame(
    StandardScaler().fit_transform(clin[numeric_cols]),
    index=clin.index,
    columns=numeric_cols
)

clin_cat = pd.get_dummies(clin[cat_cols], drop_first=True)
clin_encoded = pd.concat([clin_num, clin_cat], axis=1)
print(f"Clinical features encoded: {clin_encoded.shape[1]}")


# -------------------------------
# ENCODE LABELS
# -------------------------------
le = LabelEncoder()
labels_num = pd.Series(le.fit_transform(labels), index=labels.index, name="label")
print("Label distribution:\n", labels_num.value_counts())
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

# -------------------------------
# FINAL ALIGNMENT BEFORE CONCAT
# -------------------------------
common_final = rna_encoded.index.intersection(meth_encoded.index).intersection(clin_encoded.index).intersection(labels_num.index)
rna_encoded = rna_encoded.loc[common_final]
meth_encoded = meth_encoded.loc[common_final]
clin_encoded = clin_encoded.loc[common_final]
labels_num = labels_num.loc[common_final]

# -------------------------------
# COMBINE MODALITIES
# -------------------------------
X = pd.concat([rna_encoded, meth_encoded, clin_encoded], axis=1)

# -------------------------------
# SAVE DATA
# -------------------------------
X.to_csv("features_numeric.csv")
labels_num.to_csv("labels_numeric.csv")
print("Saved: features_numeric.csv, labels_numeric.csv")
print("Final feature matrix shape:", X.shape)

Removing duplicates in RNA: ['TCGA-06-0125', 'TCGA-14-1034']
Removing duplicates in Clinical: ['TCGA-06-0125', 'TCGA-14-0736', 'TCGA-14-1034', 'TCGA-14-1402', 'TCGA-19-0957', 'TCGA-19-1389']
RNA features encoded: 1000
Methylation features encoded: 5000


/tmp/ipython-input-3644063595.py:83: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  clin = clin.replace([np.inf, -np.inf], np.nan).fillna("-999")


Clinical features encoded: 2556
Label distribution:
 label
1    29
0    21
3    18
2    11
4     1
Name: count, dtype: int64
Label mapping: {'Classical': np.int64(0), 'Mesenchymal': np.int64(1), 'Neural': np.int64(2), 'Proneural': np.int64(3), 'nan': np.int64(4)}
Saved: features_numeric.csv, labels_numeric.csv
Final feature matrix shape: (80, 8556)


Now save and download the data in numerical format for MLP Classifier

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------------
# LOAD CLEAN NUMERIC DATA
# -------------------------------
X = pd.read_csv("features_numeric.csv", index_col=0)
y = pd.read_csv("labels_numeric.csv", index_col=0).iloc[:, 0]

print("Loaded feature matrix:", X.shape)
print("Loaded labels:", y.shape)

# -------------------------------
# ALIGN & VERIFY SAMPLE CONSISTENCY
# -------------------------------
common = sorted(list(set(X.index) & set(y.index)))
X = X.loc[common]
y = y.loc[common]

print("Samples after alignment:", len(common))

# -------------------------------
# REMOVE RARE CLASSES
# -------------------------------
min_samples_per_class = 2
counts = y.value_counts()
rare_classes = counts[counts < min_samples_per_class].index.tolist()
if rare_classes:
    print("Removing rare classes:", rare_classes)
    mask = ~y.isin(rare_classes)
    X = X.loc[mask]
    y = y.loc[mask]

print("Label distribution after removal:\n", y.value_counts())

# -------------------------------
# STRATIFIED TRAIN / VAL / TEST SPLIT
# 60% train, 20% val, 20% test
# -------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# -------------------------------
# PRINT SPLIT SIZES
# -------------------------------
print("\nFINAL SPLIT SIZES:")
print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))
print("Total:", len(X_train) + len(X_val) + len(X_test))

# -------------------------------
# SAVE SPLITS
# -------------------------------
X_train.to_csv("X_train.csv")
y_train.to_csv("y_train.csv")

X_val.to_csv("X_val.csv")
y_val.to_csv("y_val.csv")

X_test.to_csv("X_test.csv")
y_test.to_csv("y_test.csv")

print("\nSaved:")
print("  X_train.csv, y_train.csv")
print("  X_val.csv, y_val.csv")
print("  X_test.csv, y_test.csv")


Loaded feature matrix: (80, 8556)
Loaded labels: (80,)
Samples after alignment: 80
Removing rare classes: [4]
Label distribution after removal:
 label
1    29
0    21
3    18
2    11
Name: count, dtype: int64

FINAL SPLIT SIZES:
Train: 47
Validation: 16
Test: 16
Total: 79

Saved:
  X_train.csv, y_train.csv
  X_val.csv, y_val.csv
  X_test.csv, y_test.csv


Saved files:

| File                   | Description          |
| ---------------------- | -------------------- |
| `X_train.csv`          | training features    |
| `y_train.csv`          | training labels      |
| `X_val.csv`            | cross-validation     |
| `y_val.csv`           | validation labels     |
| `X_test.csv` | final test data |
| `y_test.csv`   | final test labels  |


In [ ]:
from google.colab import files

# ---- SAVE ----
X_train.to_csv("X_train.csv", index=True)
y_train.to_csv("y_train.csv", index=True)

X_val.to_csv("X_val.csv", index=True)
y_val.to_csv("y_val.csv", index=True)

X_test.to_csv("X_test.csv", index=True)
y_test.to_csv("y_test.csv", index=True)


print("Files saved. Starting downloads...")

# ---- DOWNLOAD ----
files.download("X_train.csv")
files.download("y_train.csv")

files.download("X_val.csv")
files.download("y_val.csv")

files.download("X_test.csv")
files.download("y_test.csv")


Files saved. Starting downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Note

Due to disk space and time limitations, histology embeddings will **not** be used during this phase of the project. Rather, since Methylation, mRNA sequences, and Clinical data will be used, the protocol will be steered toward **clinical-omic** fusion, prioritizing the integration of high-dimensional molecular data with low-dimensional phenotypic variables, which represents a critical real-world oncology scenario.